# Validación del Modelo Estrella Kimball — Proyecto 2 DAP
Monitoreo de Eventos de Infraestructura, Confiabilidad & Salud de Sistemas (Azure PdM)

Este notebook valida, de forma reproducible, que:
1. El DDL crea el esquema (3 hechos + 3 dimensiones) correctamente.
2. El ETL carga los CSV sin errores y sin filas huérfanas.
3. Los KPIs de confiabilidad (MTBF, MTTR, % Uptime) cuadran contra un recálculo en SQL.
4. Las consultas del DML (Window Functions) corren y devuelven resultados coherentes.

**Requisitos:** en la misma carpeta: `01_DDL_kimball.sql`, `02_etl_to_sql.py`, `03_DML_analitico.sql` y los CSV
`Dim_Machine.csv`, `Fact_SystemFailures.csv`, `Fact_TelemetryEvents.csv`, `Kpi_Reliability_by_Machine.csv`.

In [ ]:
import sqlite3, subprocess, sys, re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CARPETA = Path(".")
CSV = {n: CARPETA / f"{n}.csv" for n in
       ["Dim_Machine", "Fact_SystemFailures", "Fact_TelemetryEvents", "Kpi_Reliability_by_Machine"]}
DDL_FILE = CARPETA / "01_DDL_kimball.sql"
ETL_FILE = CARPETA / "02_etl_to_sql.py"
DML_FILE = CARPETA / "03_DML_analitico.sql"
DB_FILE  = CARPETA / "proyecto2_kimball.db"

for f in [*CSV.values(), DDL_FILE, ETL_FILE, DML_FILE]:
    print(f"{'OK   ' if f.exists() else 'FALTA'}  {f}")

## 1. Ejecutar el ETL y capturar su log

In [ ]:
# Se ejecuta como proceso aparte: valida exactamente lo que correría un evaluador
# desde la terminal (python 02_etl_to_sql.py)
resultado = subprocess.run([sys.executable, str(ETL_FILE)], cwd=CARPETA,
                           capture_output=True, text=True)
print(resultado.stdout.replace("\r", "\n")[-2500:])
if resultado.returncode != 0:
    print("=== ERROR ===")
    print(resultado.stderr)
assert resultado.returncode == 0, "El ETL terminó con error, revisar salida arriba"
print("ETL ejecutado sin errores.")

## 2. Validar el esquema (DDL)
Seis tablas esperadas y llaves foráneas declaradas en cada hecho.

In [ ]:
conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON;")

HECHOS = ["Fact_TelemetryEvents", "Fact_SystemFailures", "Fact_ReliabilityKPI"]
DIMS   = ["Dim_Calendar", "Dim_Machine", "Dim_Event"]
esperadas = set(HECHOS + DIMS)
reales = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")}
assert not (esperadas - reales), f"Faltan tablas: {esperadas - reales}"
print("Tablas presentes:", sorted(reales))

for h in HECHOS:
    print(f"\n--- FKs de {h} ---")
    for row in conn.execute(f"PRAGMA foreign_key_list({h});"):
        print(f"  {row[3]} -> {row[2]}({row[4]})")

In [ ]:
viol = conn.execute("PRAGMA foreign_key_check;").fetchall()
assert len(viol) == 0, f"Violaciones de integridad referencial: {viol[:5]}"
print("Integridad referencial: OK, 0 violaciones (PRAGMA foreign_key_check).")

## 3. Validar volumen cargado vs. CSV origen

In [ ]:
conteos = {t: conn.execute(f"SELECT COUNT(*) FROM {t};").fetchone()[0] for t in sorted(esperadas)}
display(pd.DataFrame(conteos.items(), columns=["tabla", "filas"]))

n_csv = {
    "Dim_Machine":          len(pd.read_csv(CSV["Dim_Machine"])),
    "Fact_SystemFailures":  len(pd.read_csv(CSV["Fact_SystemFailures"])),
    "Fact_ReliabilityKPI":  len(pd.read_csv(CSV["Kpi_Reliability_by_Machine"])),
    # sólo se lee una columna para no cargar 42 columnas en memoria
    "Fact_TelemetryEvents": len(pd.read_csv(CSV["Fact_TelemetryEvents"], usecols=["fact_telemetry_id"])),
}
for t, n in n_csv.items():
    assert conteos[t] == n, f"{t}: {conteos[t]} filas en BD vs {n} en CSV"
    print(f"{t:<22} coincide con el CSV origen: {n:,} filas")

## 4. FKs nulas y grano de las tablas de hechos

In [ ]:
nulos = conn.execute("""
    SELECT
      (SELECT COUNT(*) FROM Fact_TelemetryEvents WHERE machine_sk IS NULL OR calendar_sk IS NULL),
      (SELECT COUNT(*) FROM Fact_SystemFailures  WHERE machine_sk IS NULL OR calendar_sk IS NULL OR event_sk IS NULL),
      (SELECT COUNT(*) FROM Fact_ReliabilityKPI  WHERE machine_sk IS NULL)
""").fetchone()
print("FKs nulas (telemetría, eventos, KPI):", nulos)
assert all(n == 0 for n in nulos)

# Grano telemetría: 1 máquina x 1 hora, sin duplicados
dup_tel = conn.execute("""SELECT COUNT(*) FROM (SELECT machine_sk, event_ts FROM Fact_TelemetryEvents
                          GROUP BY machine_sk, event_ts HAVING COUNT(*) > 1)""").fetchone()[0]
assert dup_tel == 0, f"{dup_tel} pares (máquina, hora) duplicados en telemetría"

# Continuidad horaria: todas las máquinas con el mismo número de lecturas
huecos = pd.read_sql("""SELECT machine_sk, COUNT(*) AS n FROM Fact_TelemetryEvents GROUP BY machine_sk""", conn)
print(f"Lecturas por máquina: min={huecos.n.min()}, max={huecos.n.max()}")
assert huecos.n.nunique() == 1, "Hay máquinas con huecos en la serie horaria"

# Grano KPI: 1 fila por máquina
assert conn.execute("SELECT COUNT(DISTINCT machine_sk) FROM Fact_ReliabilityKPI").fetchone()[0] == conn.execute("SELECT COUNT(*) FROM Fact_ReliabilityKPI").fetchone()[0]
print("Grano validado: telemetría (máquina x hora) sin duplicados ni huecos, KPI 1 fila por máquina.")

## 5. Reglas de negocio
- Sensores dentro de rangos físicos plausibles y sin nulos (soberanía del dato: no se eliminan outliers).
- Flags de anomalía sólo 0/1 y consistentes con el consolidado.
- Lags NULL únicamente en las primeras 1 h / 3 h de cada máquina.
- Eventos simultáneos preservados (no son duplicados).
- `downtime_hrs` coherente con el tipo de evento (12 falla / 6 mantenimiento / 0 error).

In [ ]:
chk = conn.execute("""
SELECT
  SUM(volt IS NULL OR rotate IS NULL OR pressure IS NULL OR vibration IS NULL),
  SUM(volt <= 0 OR rotate <= 0 OR pressure <= 0 OR vibration <= 0),
  SUM(event_hour NOT BETWEEN 0 AND 23),
  SUM(is_sensorial_anomaly NOT IN (0,1) OR is_sensorial_anomaly_z NOT IN (0,1)),
  SUM(is_sensorial_anomaly <> MAX(is_volt_anomaly_iqr, is_rotate_anomaly_iqr, is_pressure_anomaly_iqr, is_vibration_anomaly_iqr)),
  SUM(is_sensorial_anomaly_z <> MAX(is_volt_anomaly_z, is_rotate_anomaly_z, is_pressure_anomaly_z, is_vibration_anomaly_z)),
  SUM(volt_lag_1h IS NULL), SUM(volt_lag_3h IS NULL)
FROM Fact_TelemetryEvents
""").fetchone()
etiquetas = ["sensores nulos", "sensores <= 0", "hora fuera de 0-23", "flags no binarias",
             "flag IQR inconsistente", "flag Z inconsistente", "lag_1h nulos", "lag_3h nulos"]
display(pd.DataFrame({"chequeo": etiquetas, "filas": chk}))
assert chk[:6] == (0,)*6, "Fallan reglas de negocio en telemetría"
n_maq = conn.execute("SELECT COUNT(*) FROM Dim_Machine").fetchone()[0]
assert chk[6] == n_maq and chk[7] == 3*n_maq, "Los NULL de lag deben ser 1 y 3 por máquina"

dt = pd.read_sql("""SELECT e.event_type, f.downtime_hrs, COUNT(*) n
                    FROM Fact_SystemFailures f JOIN Dim_Event e ON f.event_sk = e.event_sk
                    GROUP BY e.event_type, f.downtime_hrs""", conn)
display(dt)
esperado = {"failure": 12.0, "maintenance": 6.0, "error": 0.0}
assert all(esperado[r.event_type] == r.downtime_hrs for r in dt.itertuples()), "downtime_hrs incoherente"

simult = conn.execute("""SELECT COUNT(*) FROM (SELECT machine_sk, event_ts FROM Fact_SystemFailures
                         GROUP BY machine_sk, event_ts HAVING COUNT(*) > 1)""").fetchone()[0]
print(f"Instantes con eventos simultáneos preservados: {simult} (legítimos, no se deduplican)")

## 6. KPIs de confiabilidad: BD vs. CSV de Python

In [ ]:
kpi_csv = pd.read_csv(CSV["Kpi_Reliability_by_Machine"]).set_index("machineID").sort_index()
kpi_db = pd.read_sql("""SELECT m.machine_id, k.* FROM Fact_ReliabilityKPI k
                        JOIN Dim_Machine m ON k.machine_sk = m.machine_sk""", conn).set_index("machine_id").sort_index()
for a, b in [("n_failures","n_failures"), ("n_maint_events","n_maint_events"),
             ("MTBF_hours","mtbf_hours"), ("MTTR_hours","mttr_hours"), ("pct_uptime","pct_uptime")]:
    dif = (kpi_csv[a] - kpi_db[b]).abs().max()
    print(f"{b:<16} diferencia máxima vs CSV: {dif:.2e}")
    assert dif < 1e-6
print("\nKPIs cargados idénticos al CSV origen.")

## 7. Ejecutar el DML completo y mostrar cada resultado

In [ ]:
def dividir_sql(texto):
    """Separa sentencias por ';' ignorando bloques que sólo tienen comentarios."""
    out = []
    for p in (x.strip() for x in texto.split(";")):
        if any(l.strip() and not l.strip().startswith("--") for l in p.splitlines()):
            out.append(p)
    return out

def titulo(q):
    m = re.search(r"--\s*(Consulta\s+\d+:[^\n]*)", q)
    return m.group(1).strip() if m else "(sin título)"

consultas = dividir_sql(DML_FILE.read_text(encoding="utf-8"))
print(f"{len(consultas)} consultas detectadas en {DML_FILE.name}\n")
R, errores = {}, []
for i, q in enumerate(consultas):
    try:
        R[i] = pd.read_sql(q, conn)
        print(f"OK     {titulo(q)}  ->  {R[i].shape[0]} filas x {R[i].shape[1]} cols")
    except Exception as e:
        errores.append(i); print(f"FALLA  {titulo(q)}  ->  {e}")
assert not errores, f"Consultas con error: {errores}"

### 7.1 Tarjetas ejecutivas y reconciliación de KPIs

In [ ]:
display(R[0])
rec = R[10]
display(rec)
assert rec.loc[0, ["dif_n_failures", "dif_n_maint"]].sum() == 0
assert rec.loc[0, ["max_dif_downtime_hrs", "max_dif_uptime_pct", "max_dif_mtbf_hrs", "max_dif_mttr_hrs"]].max() < 1e-6
print("KPIs recalculados en SQL desde los hechos == KPIs de Python.")
print(f"Nota: {int(rec.loc[0,'mant_previos_a_ventana'])} mantenimientos ocurren ANTES de la ventana de telemetría "
      "pero descuentan downtime en el KPI (ver nota de auditoría en el README / informe).")

### 7.2 Confiabilidad por modelo y banda de edad

In [ ]:
display(R[1])

### 7.3 Semáforo de riesgo (RAG) — matriz de equipos

In [ ]:
rag = R[2]
display(rag.head(10))
print(rag["rag_status"].value_counts().to_string())

colores = {"ROJO": "#d62728", "AMBAR": "#ffbf00", "VERDE": "#2ca02c"}
orden = rag.sort_values("machine_id")
fig, ax = plt.subplots(figsize=(11, 4))
for i, r in enumerate(orden.itertuples()):
    ax.add_patch(plt.Rectangle((i % 20, -(i // 20)), 0.9, 0.9, color=colores[r.rag_status]))
    ax.text(i % 20 + 0.45, -(i // 20) + 0.45, r.machine_id, ha="center", va="center", fontsize=7, color="white")
ax.set_xlim(0, 20); ax.set_ylim(-(len(orden) // 20) , 1); ax.axis("off")
ax.set_title("Matriz de equipos por semáforo de riesgo (RAG)")
plt.show()

### 7.4 Tiempo entre fallas (LAG / LEAD)

In [ ]:
display(R[3].head(12))

### 7.5 Degradación de parámetros vs. ocurrencia de falla

In [ ]:
deg = R[4]
display(deg)
orden_v = sorted(deg["ventana_pre_falla"].unique(), reverse=True)  # de lejos a cerca de la falla
fig, ax = plt.subplots(figsize=(9, 4))
for comp, g in deg.groupby("component"):
    g = g.set_index("ventana_pre_falla").reindex(orden_v)
    ax.plot(range(len(g)), g["z_vibration"], marker="o", label=comp)
ax.set_xticks(range(len(orden_v))); ax.set_xticklabels(orden_v, rotation=15)
ax.axhline(0, color="gray", lw=0.8); ax.set_ylabel("z-score de vibración")
ax.set_title("Vibración vs. horas antes de la falla (por componente)"); ax.legend()
plt.show()

### 7.6 Tarjetas de alerta temprana

In [ ]:
alertas = R[5]
display(alertas[alertas["estado_alerta"] != "Normal"].head(15))
print(alertas["estado_alerta"].value_counts().to_string())

### 7.7 Pareto de fallas por componente

In [ ]:
display(R[6])

### 7.8 Mantenimiento previo a cada falla

In [ ]:
display(R[7])

### 7.9 Serie diaria con promedio móvil de 7 días

In [ ]:
s = R[8].copy(); s["date_key"] = pd.to_datetime(s["date_key"])
ax = s.set_index("date_key")[["fallas", "ma7_fallas"]].plot(figsize=(11, 4), title="Fallas diarias vs. promedio móvil 7 días")
ax.set_xlabel("Fecha"); ax.set_ylabel("Fallas")
plt.show()

### 7.10 Heatmap de fallas (día de la semana x hora)

In [ ]:
h = R[9].pivot(index="day_of_week", columns="event_hour", values="fallas").fillna(0)
h = h.reindex(["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"])
display(h.style.background_gradient(cmap="Oranges", axis=None))

### 7.11 Auditoría de calidad de datos

In [ ]:
display(R[11])
assert R[11].loc[0, "maquinas_con_huecos_horarios"] == 0

## 8. Resumen final de la validación

In [ ]:
print("=" * 62)
print("RESUMEN DE VALIDACIÓN - MODELO KIMBALL (PROYECTO 2)")
print("=" * 62)
print(f"Tablas creadas              : {len(reales)} / 6 esperadas")
for t in HECHOS: print(f"Filas {t:<22}: {conteos[t]:,}")
print(f"Violaciones de FK           : {len(viol)} (esperado: 0)")
print(f"FKs nulas en hechos         : {sum(nulos)} (esperado: 0)")
print(f"Consultas DML ejecutadas    : {len(R)} / {len(consultas)}")
print("=" * 62)
print("Validación completa: el modelo cumple integridad, los KPIs reconcilian")
print("y las consultas con Window Functions corren sobre proyecto2_kimball.db")
conn.close()